<a href="https://colab.research.google.com/drive/1HLxEVsGfXvEm73Rszwd-ucUW6wkQtE9e?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

### Generative Agents

In [1]:
!pip install -qU google-generativeai


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import google.generativeai as genai
import getpass
from datetime import datetime

Get free-tier Google's Gemini API Key here: https://aistudio.google.com/app/apikey

In [3]:
# Prefer an environment variable, fall back to prompting.
# The prompt alone meant these notebooks could not run non-interactively
# (nbconvert, papermill, CI) and made you retype the key once per notebook.
import os
API_KEY = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    API_KEY = getpass.getpass("Enter your Google API key: ")

In [4]:
genai.configure(api_key=API_KEY)

In [5]:
class GenerativeAgent:
    def __init__(self, name):
        self.model = genai.GenerativeModel("gemini-flash-latest")
        self.name = name
        self.memory_stream = []  # All observations
        self.reflections = []    # High-level insights
        self.environment = {}    # Current environment state

    def observe(self, observation):
        """Record observation in memory stream"""
        memory = {
            "time": datetime.now().strftime("%H:%M"),
            "type": "observation",
            "content": observation
        }
        self.memory_stream.append(memory)
        print(f" {self.name} observed: {observation}")

    def reflect(self):
        """Generate insights from recent memories"""
        if len(self.memory_stream) < 3:
            return

        recent = "\n".join([m["content"] for m in self.memory_stream[-5:]])

        prompt = f"""Based on these observations, generate 2-3 high-level insights:

        {recent}

        Insights:"""

        response = self.model.generate_content(prompt).text

        reflection = {
            "time": datetime.now().strftime("%H:%M"),
            "type": "reflection",
            "content": response.strip()
        }
        self.reflections.append(reflection)
        print(f"{self.name} reflected: {response.strip()}\n")

    def plan(self, goal):
        """Create action plan based on memories and environment"""
        memories = "\n".join([m["content"] for m in self.memory_stream[-3:]])
        insights = "\n".join([r["content"] for r in self.reflections[-2:]])
        env = str(self.environment)

        prompt = f"""You are {self.name}. Create a short action plan.

        Goal: {goal}

        Recent memories:
        {memories}

        Insights:
        {insights if insights else "None yet"}

        Environment:
        {env}

        Plan (3-5 actions):"""

        plan = self.model.generate_content(prompt).text
        print(f"{self.name}'s plan:\n{plan}\n")
        return plan.strip()

    def act(self, action, environment):
        """Take action and update environment"""
        print(f"{self.name} action: {action}")

        # Simulate action effect on environment
        prompt = f"""An agent performed this action: {action}

        Current environment:
        {environment}

        How does the environment change? Respond with key-value pairs.
        Format: key: new_value

        Changes:"""

        response = self.model.generate_content(prompt).text

        # Parse environment changes
        for line in response.split("\n"):
            if ":" in line:
                key, val = line.split(":", 1)
                environment[key.strip()] = val.strip()

        print(f"Environment updated: {environment}\n")
        return environment

    def run_cycle(self, goal, steps=5):
        """Run observe-reflect-plan-act cycle"""
        print(f"\n{'='*60}")
        print(f"{self.name}: {goal}")
        print(f"{'='*60}\n")

        for i in range(steps):
            print(f"--- Cycle {i+1} ---\n")

            # Observe environment
            self.observe(f"Environment state: {self.environment}")

            # Reflect periodically
            if i > 0 and i % 2 == 0:
                self.reflect()

            # Plan next action
            plan = self.plan(goal)

            # Extract first action from plan
            action = plan.split("\n")[0].strip()

            # Act and update environment
            self.environment = self.act(action, self.environment)

            # Record action in memory
            self.memory_stream.append({
                "time": datetime.now().strftime("%H:%M"),
                "type": "action",
                "content": action
            })

        print(f"[OK] Goal completed: {goal}\n")
        self.show_summary()

    def show_summary(self):
        """Show agent's memory and insights"""
        print(f"{'='*60}")
        print(f"{self.name}'s Summary")
        print(f"{'='*60}")
        print(f"Memories: {len(self.memory_stream)}")
        print(f"Reflections: {len(self.reflections)}")
        print(f"Final environment: {self.environment}")

        if self.reflections:
            print(f"\nKey insights:")
            for r in self.reflections[-2:]:
                print(f"  • {r['content'][:80]}...")

In [6]:
# Example 1: Research Assistant Agent
print("="*60)
print("EXAMPLE 1: Research Assistant Agent")
print("="*60)

researcher = GenerativeAgent("ResearchBot")
researcher.environment = {
    "papers_read": 0,
    "notes": "empty",
    "draft": "not started"
}

researcher.run_cycle("Research AI agents and write summary", steps=4)


# Example 2: Customer Service Agent
print("\n" + "="*60)
print("EXAMPLE 2: Customer Service Agent")
print("="*60)

cs_agent = GenerativeAgent("SupportBot")
cs_agent.environment = {
    "tickets": 5,
    "customer_satisfaction": "unknown",
    "resolved": 0
}

cs_agent.run_cycle("Handle customer support tickets", steps=3)


# Example 3: Multi-agent simulation
print("\n" + "="*60)
print("EXAMPLE 3: Multi-Agent Collaboration")
print("="*60)

manager = GenerativeAgent("Manager")
developer = GenerativeAgent("Developer")

# Shared environment
shared_env = {
    "project_status": "not started",
    "tasks": 10,
    "completed": 0
}

manager.environment = shared_env.copy()
developer.environment = shared_env.copy()

print("--- Manager's turn ---")
manager.observe("New project assigned with 10 tasks")
plan = manager.plan("Organize project and assign tasks")
action = plan.split("\n")[0]
shared_env = manager.act(action, shared_env)

print("\n--- Developer's turn ---")
developer.environment = shared_env.copy()
developer.observe(f"Manager assigned tasks: {shared_env}")
dev_plan = developer.plan("Complete assigned tasks")
dev_action = dev_plan.split("\n")[0]
shared_env = developer.act(dev_action, shared_env)

print(f"\nFinal shared environment: {shared_env}")

EXAMPLE 1: Research Assistant Agent

ResearchBot: Research AI agents and write summary

--- Cycle 1 ---

 ResearchBot observed: Environment state: {'papers_read': 0, 'notes': 'empty', 'draft': 'not started'}


ResearchBot's plan:
**Plan:**

1. **Search & Select Literature:** Query databases for 3–5 foundational and recent papers on AI agent architectures and capabilities.
2. **Read & Extract Insights:** Read the selected papers and compile structured notes on key methodologies, frameworks, and benchmarks.
3. **Draft Summary:** Synthesize the notes into a structured summary covering definitions, current trends, and challenges in AI agents.
4. **Review & Refine:** Review the draft for clarity, accuracy, and completeness against the research goal.

ResearchBot action: **Plan:**


Environment updated: {'papers_read': 0, 'notes': 'empty', 'draft': 'not started'}

--- Cycle 2 ---

 ResearchBot observed: Environment state: {'papers_read': 0, 'notes': 'empty', 'draft': 'not started'}


ResearchBot's plan:
1. Search and collect 3-5 foundational and recent research papers on AI agents.
2. Read the collected papers and extract key insights, architectures, and findings into notes.
3. Synthesize the notes to create an outline and write the initial summary draft.
4. Review, refine, and finalize the summary report on AI agents.

ResearchBot action: 1. Search and collect 3-5 foundational and recent research papers on AI agents.


Environment updated: {'papers_read': 0, 'notes': 'empty', 'draft': 'not started', 'papers_collected': '3-5 foundational and recent research papers on AI agents'}

--- Cycle 3 ---

 ResearchBot observed: Environment state: {'papers_read': 0, 'notes': 'empty', 'draft': 'not started', 'papers_collected': '3-5 foundational and recent research papers on AI agents'}


...

Environment updated: {'tickets': '3', 'customer_satisfaction': 'satisfied', 'resolved': '2'}

[OK] Goal completed: Handle customer support tickets

SupportBot's Summary
Memories: 6
Reflections: 1
Final environment: {'tickets': '3', 'customer_satisfaction': 'satisfied', 'resolved': '2'}

Key insights:
  • Here are 3 high-level insights based on the observations:

1. **Immediate Positi...

EXAMPLE 3: Multi-Agent Collaboration
--- Manager's turn ---
 Manager observed: New project assigned with 10 tasks


Manager's plan:
1. Review and prioritize the 10 tasks based on dependencies and urgency.
2. Evaluate team capacity and assign tasks to appropriate team members.
3. Establish clear deadlines and milestones for each task.
4. Conduct a project kickoff meeting to align expectations and start execution.

Manager action: 1. Review and prioritize the 10 tasks based on dependencies and urgency.


Environment updated: {'project_status': 'in progress', 'tasks': 10, 'completed': 0}


--- Developer's turn ---
 Developer observed: Manager assigned tasks: {'project_status': 'in progress', 'tasks': 10, 'completed': 0}


Developer's plan:
**Action Plan:**

1. **Review and Prioritize:** Analyze the 10 assigned tasks, identify dependencies, and prioritize them by urgency and impact.
2. **Execute Development:** Implement the tasks sequentially according to priority, writing clean and modular code.
3. **Test and Verify:** Run local unit and integration tests to ensure each completed task meets requirements and does not introduce regressions.
4. **Submit and Report:** Create pull requests for completed work, update task statuses in the tracker, and report progress to the Manager.

Developer action: **Action Plan:**


Environment updated: {'project_status': 'in progress', 'tasks': 10, 'completed': 0}


Final shared environment: {'project_status': 'in progress', 'tasks': 10, 'completed': 0}
